# Chapitre 4 - Méthodes linéaires de classification
## Cours 3 : régression logistique et hyperplans séparateurs

Ce notebook couvre les sections 4.4 et 4.5 du livre :

1. le **modèle logistique** et son ajustement par maximum de vraisemblance via
   l'algorithme de **Newton-Raphson**, qui prend la forme de **moindres carrés pondérés
   itérés** (IRLS) ;
2. l'exemple des **maladies cardiaques en Afrique du Sud** (figure 4.12, tableaux 4.2 et
   4.3) et l'inférence associée ;
3. la **régression logistique pénalisée L1** (figure 4.13) ;
4. la comparaison LDA / logistique ;
5. les **hyperplans séparateurs** : perceptron de Rosenblatt (figure 4.14) et hyperplan
   séparateur optimal (figure 4.16).

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
from esl import datasets
from esl import ch04_classification as c4
set_style()

## 1. Le modèle logistique (section 4.4)

On modélise directement les probabilités a posteriori $\Pr(G = k \mid X = x)$ par des
fonctions de $x$ dont les **log-rapports** à une classe de référence (la dernière) sont
affines (éq. (4.17) du livre) :
$$\log\frac{\Pr(G=k\mid X=x)}{\Pr(G=K\mid X=x)} = \beta_{k0} + \beta_k^Tx, \qquad k = 1,\dots,K-1.$$

En exponentiant et en imposant que les $K$ probabilités somment à 1, on obtient
(éq. (4.18)) :
$$\Pr(G=k\mid X=x) = \frac{\exp(\beta_{k0}+\beta_k^Tx)}{1+\sum_{\ell=1}^{K-1}\exp(\beta_{\ell 0}+\beta_\ell^Tx)}
\quad (k<K), \qquad
\Pr(G=K\mid X=x) = \frac{1}{1+\sum_{\ell=1}^{K-1}\exp(\beta_{\ell 0}+\beta_\ell^Tx)}.$$
Ces quantités sont bien positives et de somme 1. Le choix de la classe de référence est
arbitraire : les estimations des probabilités n'en dépendent pas.

**Deux classes.** On code la réponse $y_i = 1$ pour la classe 1 et $y_i = 0$ pour la
classe 2, et on note $p(x;\beta) = \Pr(G=1\mid X=x)$. Avec $x$ complété d'un 1 initial
(pour la constante) et $\beta = (\beta_{10}, \beta_1)$ :
$$p(x;\beta) = \sigma(\beta^Tx), \qquad \sigma(t) = \frac{e^t}{1+e^t} \quad\text{(fonction sigmoïde)}.$$

### Vraisemblance

Les $y_i$ étant des variables de Bernoulli indépendantes (conditionnellement aux $x_i$)
de paramètres $p(x_i;\beta)$, la log-vraisemblance vaut (éq. (4.20))
$$\ell(\beta) = \sum_{i=1}^N\Big[y_i\log p(x_i;\beta) + (1-y_i)\log(1-p(x_i;\beta))\Big]
= \sum_{i=1}^N\Big[y_i\beta^Tx_i - \log\big(1+e^{\beta^Tx_i}\big)\Big].$$
Pour la seconde forme, on a utilisé $\log p = \beta^Tx - \log(1+e^{\beta^Tx})$ et
$\log(1-p) = -\log(1+e^{\beta^Tx})$.

### Gradient et hessienne

On dérive terme à terme. Comme $\frac{d}{dt}\log(1+e^t) = \sigma(t)$, et par la règle de
dérivation composée,
$$\frac{\partial\ell}{\partial\beta} = \sum_{i=1}^N x_i\big(y_i - p(x_i;\beta)\big)
\qquad\text{(éq. (4.21))}.$$
En dérivant encore, avec $\sigma'(t) = \sigma(t)(1-\sigma(t))$ :
$$\frac{\partial^2\ell}{\partial\beta\,\partial\beta^T} = -\sum_{i=1}^N x_ix_i^T\,p(x_i;\beta)\big(1-p(x_i;\beta)\big)
\qquad\text{(éq. (4.22))}.$$

> **Concavité.** Pour tout vecteur $v$, $v^T\big(\partial^2\ell\big)v = -\sum_i p_i(1-p_i)(v^Tx_i)^2 \le 0$ :
> la hessienne est semi-définie négative, donc $\ell$ est **concave**. Un point où le
> gradient s'annule est un maximum global ; il est unique si $\mathbf{X}$ est de rang plein
> et si ce maximum existe (il n'existe pas toujours : voir l'exercice 4.5).

### Newton-Raphson et moindres carrés pondérés itérés

L'algorithme de Newton remplace $\ell$ par son développement de Taylor d'ordre 2 autour de
l'itéré courant et maximise cette approximation quadratique (éq. (4.23)) :
$$\beta^{\text{new}} = \beta^{\text{old}} - \Big(\frac{\partial^2\ell}{\partial\beta\,\partial\beta^T}\Big)^{-1}\frac{\partial\ell}{\partial\beta}.$$
En notation matricielle, avec $\mathbf{p}$ le vecteur des $p(x_i;\beta^{\text{old}})$ et
$\mathbf{W} = \operatorname{diag}\big(p_i(1-p_i)\big)$, le gradient vaut
$\mathbf{X}^T(\mathbf{y}-\mathbf{p})$ et la hessienne $-\mathbf{X}^T\mathbf{W}\mathbf{X}$.

Le pas de Newton se réécrit (éq. (4.26)) :
$$\begin{aligned}
\beta^{\text{new}} &= \beta^{\text{old}} + (\mathbf{X}^T\mathbf{W}\mathbf{X})^{-1}\mathbf{X}^T(\mathbf{y}-\mathbf{p}) \\
&= (\mathbf{X}^T\mathbf{W}\mathbf{X})^{-1}\mathbf{X}^T\mathbf{W}\big(\mathbf{X}\beta^{\text{old}} + \mathbf{W}^{-1}(\mathbf{y}-\mathbf{p})\big)
= (\mathbf{X}^T\mathbf{W}\mathbf{X})^{-1}\mathbf{X}^T\mathbf{W}\mathbf{z},
\end{aligned}$$
où l'on a factorisé $(\mathbf{X}^T\mathbf{W}\mathbf{X})^{-1}\mathbf{X}^T\mathbf{W}$ et
introduit la **réponse ajustée** $\mathbf{z} = \mathbf{X}\beta^{\text{old}} + \mathbf{W}^{-1}(\mathbf{y}-\mathbf{p})$.

On reconnaît la solution des **moindres carrés pondérés**
$\beta^{\text{new}} = \operatorname*{arg\,min}_\beta (\mathbf{z}-\mathbf{X}\beta)^T\mathbf{W}(\mathbf{z}-\mathbf{X}\beta)$
(éq. (4.28)). Comme $\mathbf{p}$, $\mathbf{W}$ et $\mathbf{z}$ changent à chaque itération,
on parle de **moindres carrés pondérés itérés** (IRLS). On part en général de $\beta = 0$ ;
la convergence n'est pas garantie en théorie (la log-vraisemblance étant concave, elle
l'est presque toujours en pratique ; sinon on réduit le pas).

## 2. Exemple : maladies cardiaques en Afrique du Sud (section 4.4.2)

Les données proviennent d'une enquête sur les facteurs de risque coronarien dans trois
régions rurales du Cap-Occidental. La réponse `chd` indique un infarctus du myocarde
(160 cas, 302 témoins). Les prédicteurs : pression systolique `sbp`, tabac cumulé en kg
`tobacco`, cholestérol LDL `ldl`, `adiposity`, antécédents familiaux `famhist` (0/1),
score de comportement de type A `typea`, `obesity`, `alcohol` et `age`.

In [2]:
sa = datasets.load_saheart()
print(sa.shape)
sa.head()

(462, 10)


,sbp,tobacco,ldl,adiposity,famhist,typea,obesity,alcohol,age,chd
0,160,12.00,5.73,23.11,1,49,25.30,97.20,52,1
1,144,0.01,4.41,28.61,0,55,28.87,2.06,63,1
2,118,0.08,3.48,32.28,1,52,29.14,3.81,46,0
3,170,7.50,6.41,38.03,1,51,31.99,24.26,58,1
4,134,13.60,3.50,27.78,1,60,25.99,57.34,49,1
